In [1]:
import os
import ast
import pandas as pd
from getpass import getpass
from openai import OpenAI

# =========================
# PATHS
# =========================
INPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/Zero-Shot/zero_shot_results"
OUTPUT_DIR = "/Users/shinekhantaung/Desktop/GPT-NER/Zero-Shot/SV-ZeroShot"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# API
# =========================
api_key = getpass("Enter OpenAI API Key: ")
client = OpenAI(api_key=api_key)

# =========================
# HELPERS
# =========================
def parse_names(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x

    x = str(x).strip()

    if x.startswith("["):
        try:
            return ast.literal_eval(x)
        except:
            return []

    if ";" in x:
        return [i.strip() for i in x.split(";") if i.strip()]

    if x == "" or x.lower() == "none":
        return []

    return [x]

def verify_name(sentence, word):
    prompt = f"""
Sentence: {sentence}

Word: {word}

Is "{word}" referring to a person name in this sentence?
Answer only Yes or No.
"""
    response = client.chat.completions.create(
        model="gpt-4.1",
        temperature=0,
        messages=[{"role": "user", "content": prompt}]
    )
    ans = response.choices[0].message.content.strip().lower()
    return "yes" in ans

def compute_metrics(preds, golds):
    TP = FP = FN = 0

    for p, g in zip(preds, golds):
        p_set = set(p)
        g_set = set(g)

        TP += len(p_set & g_set)
        FP += len(p_set - g_set)
        FN += len(g_set - p_set)

    precision = TP / (TP + FP) if TP + FP > 0 else 0
    recall    = TP / (TP + FN) if TP + FN > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0

    return precision, recall, f1

# =========================
# SELF-VERIFICATION PER FOLD
# =========================
all_results = []

for fold in range(1, 6):
    print(f"\n===== Processing Fold {fold} =====")

    input_file = os.path.join(INPUT_DIR, f"predictions_fold_{fold}.xlsx")
    df = pd.read_excel(input_file)

    sv_results = []

    for _, row in df.iterrows():
        sentence = row["sentence"]
        pred_names = parse_names(row["pred_names_parsed"])

        verified = []
        for name in pred_names:
            try:
                if verify_name(sentence, name):
                    verified.append(name)
            except Exception as e:
                print(f"Fold {fold} error on name '{name}': {e}")

        sv_results.append(verified)

    df["SV_pred_names"] = sv_results

    gold = df["gold_names_parsed"].apply(parse_names)
    pred_after = df["SV_pred_names"]

    P, R, F1 = compute_metrics(pred_after, gold)

    print(f"Fold {fold} -> Precision: {P:.4f}, Recall: {R:.4f}, F1: {F1:.4f}")

    all_results.append({
        "fold": fold,
        "precision": P,
        "recall": R,
        "f1_score": F1
    })

    output_file = os.path.join(OUTPUT_DIR, f"SV-ZeroShot_fold_{fold}.xlsx")
    df.to_excel(output_file, index=False)

# =========================
# SAVE METRICS SUMMARY
# =========================
summary_df = pd.DataFrame(all_results)

avg_row = {
    "fold": "avg",
    "precision": summary_df["precision"].mean(),
    "recall": summary_df["recall"].mean(),
    "f1_score": summary_df["f1_score"].mean()
}

summary_df = pd.concat([summary_df, pd.DataFrame([avg_row])], ignore_index=True)

summary_file = os.path.join(OUTPUT_DIR, "SV-ZeroShot_metrics_summary.xlsx")
summary_df.to_excel(summary_file, index=False)

print("\nSaved fold files to:", OUTPUT_DIR)
print("Saved metrics summary to:", summary_file)
print("\nFinal Average F1:", avg_row["f1_score"])


===== Processing Fold 1 =====
Fold 1 -> Precision: 0.9205, Recall: 0.8088, F1: 0.8611

===== Processing Fold 2 =====
Fold 2 -> Precision: 0.9148, Recall: 0.8226, F1: 0.8662

===== Processing Fold 3 =====
Fold 3 -> Precision: 0.9032, Recall: 0.8485, F1: 0.8750

===== Processing Fold 4 =====
Fold 4 -> Precision: 0.9148, Recall: 0.8226, F1: 0.8662

===== Processing Fold 5 =====
Fold 5 -> Precision: 0.8996, Recall: 0.8240, F1: 0.8601

Saved fold files to: /Users/shinekhantaung/Desktop/GPT-NER/Zero-Shot/SV-ZeroShot
Saved metrics summary to: /Users/shinekhantaung/Desktop/GPT-NER/Zero-Shot/SV-ZeroShot/SV-ZeroShot_metrics_summary.xlsx

Final Average F1: 0.8657332177722331
